# StressDetection — Training on Google Colab

This notebook walks through the **complete training pipeline** for the
Resilience-First AI Stress Detection system.

### Key features
- **Google Drive persistence** — raw datasets, processed data, and trained
  checkpoints are saved to your Google Drive so nothing is lost when the
  Colab session ends.
- **Multi-dataset support** — merges Dreaddit, Reddit_Combi, Reddit_Title,
  Twitter_Full, Stressed_Tweets, and optional Happy_Neutral into a single
  unified dataset.
- **Automatic resume** — if you restart a session, the notebook detects
  previously processed data and checkpoints on Drive and skips redundant steps.

**Before you start:**
1. Go to **Runtime → Change runtime type** and select **T4 GPU**.
2. Have your dataset files ready (see Step 3).

After training you will have `model.pt` saved to Google Drive,
which is all you need to run the application on your Windows 11 machine.

---
**Tokenizer note:** The tokenizer uses `hashlib.md5` (not Python's `hash()`),
so token IDs are identical on every platform, regardless of `PYTHONHASHSEED`.

## Step 1 — Mount Google Drive

All data and checkpoints will be stored on your Google Drive under
`MyDrive/StressDetection/`. This ensures nothing is lost when the
Colab runtime disconnects or restarts.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ── Persistent storage paths on Google Drive ──
DRIVE_BASE       = '/content/drive/MyDrive/StressDetection'
DRIVE_RAW        = os.path.join(DRIVE_BASE, 'data', 'raw')
DRIVE_PROCESSED  = os.path.join(DRIVE_BASE, 'data', 'processed')
DRIVE_CHECKPOINTS = os.path.join(DRIVE_BASE, 'checkpoints')
DRIVE_LOGS       = os.path.join(DRIVE_BASE, 'logs')

for d in [DRIVE_RAW, DRIVE_PROCESSED, DRIVE_CHECKPOINTS, DRIVE_LOGS]:
    os.makedirs(d, exist_ok=True)

print(f'Drive base: {DRIVE_BASE}')
print('Directories created:')
for d in [DRIVE_RAW, DRIVE_PROCESSED, DRIVE_CHECKPOINTS, DRIVE_LOGS]:
    print(f'  {d}')

## Step 2 — Clone Repository & Install Dependencies

In [ ]:
# Clone the StressDetection repository (or update if already cloned)
REPO_DIR = '/content/StressDetection'

if os.path.isdir(REPO_DIR):
    print('Repository already cloned — pulling latest changes...')
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/anant-925/StressDetection.git {REPO_DIR}

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Training will be slow on CPU.')
    print('Go to Runtime > Change runtime type > T4 GPU')

## Step 3 — Upload Raw Datasets to Google Drive

Place the following files in `data/raw/` on Google Drive.
If you already uploaded them in a previous session, they are still
there — skip this cell.

| File | Domain |
|------|--------|
| `dreaddit-train.csv` (or `.csv.zip`) | Reddit Long |
| `Reddit_Combi.csv` (or `.xlsx`) | Reddit Long |
| `Reddit_Title.csv` (or `.xlsx`) | Reddit Short |
| `Twitter_Full.csv` (or `.xlsx`) | Twitter Short |
| `Stressed_Tweets.csv` | Twitter Short (implicit label=1) |
| `Happy_Neutral.csv` | Optional negatives (label=0) |

**Option A — Upload from your computer** (files are saved to Drive automatically):

In [ ]:
import shutil
from google.colab import files

# Check if datasets already exist on Drive
existing = os.listdir(DRIVE_RAW) if os.path.isdir(DRIVE_RAW) else []
if existing:
    print('Files already on Google Drive (data/raw/):')
    for f in sorted(existing):
        size_mb = os.path.getsize(os.path.join(DRIVE_RAW, f)) / 1e6
        print(f'  {f:40s}  {size_mb:.1f} MB')
    print('\nTo re-upload, delete the files from Drive first.')
    print('Otherwise, skip this cell and proceed to Step 4.')
else:
    print('Select your dataset files in the dialog that appears...')
    uploaded = files.upload()
    for filename in uploaded:
        dest = os.path.join(DRIVE_RAW, filename)
        shutil.move(filename, dest)
        print(f'  Saved {filename} -> {dest}')
    print(f'\n{len(uploaded)} file(s) saved to Google Drive.')

**Option B — Copy from another Drive folder** (run this instead of Option A):

In [ ]:
# ── Option B: Copy from another Google Drive folder ──
# Uncomment and adjust the source path.

# SOURCE_DIR = '/content/drive/MyDrive/datasets'  # <-- change this path
#
# for fname in os.listdir(SOURCE_DIR):
#     src = os.path.join(SOURCE_DIR, fname)
#     dst = os.path.join(DRIVE_RAW, fname)
#     if not os.path.isfile(dst):
#         shutil.copy2(src, dst)
#         print(f'Copied {fname}')
#     else:
#         print(f'Skipped {fname} (already exists)')
print('Option B cell — uncomment and edit the source path, then run.')

In [ ]:
# Sync Drive raw datasets to local repo data/raw/ (used by data_preprocessing.py)
import shutil

LOCAL_RAW = os.path.join(REPO_DIR, 'data', 'raw')
os.makedirs(LOCAL_RAW, exist_ok=True)

for fname in os.listdir(DRIVE_RAW):
    src = os.path.join(DRIVE_RAW, fname)
    dst = os.path.join(LOCAL_RAW, fname)
    if os.path.isfile(src):
        shutil.copy2(src, dst)

raw_files = os.listdir(LOCAL_RAW)
print(f'Files in data/raw/ ({len(raw_files)}):')  
for f in sorted(raw_files):
    size_mb = os.path.getsize(os.path.join(LOCAL_RAW, f)) / 1e6
    print(f'  {f:40s}  {size_mb:.1f} MB')

if not raw_files:
    print('WARNING: No dataset files found. Go back to Step 3.')

## Step 4 — Preprocess Data (Multi-Dataset Merge)

Merges all uploaded datasets into a single unified CSV at
`data/processed/unified_stress.csv` with columns `text`, `label`, `domain`,
plus any numeric feature columns found in the source data.

The processed file is also saved to Google Drive for persistence.

In [ ]:
import shutil

PROCESSED_CSV = os.path.join(REPO_DIR, 'data', 'processed', 'unified_stress.csv')
DRIVE_PROCESSED_CSV = os.path.join(DRIVE_PROCESSED, 'unified_stress.csv')

# Check if a previously processed file exists on Drive
if os.path.isfile(DRIVE_PROCESSED_CSV):
    print('Found previously processed data on Google Drive.')
    print('Copying to local workspace...')
    os.makedirs(os.path.dirname(PROCESSED_CSV), exist_ok=True)
    shutil.copy2(DRIVE_PROCESSED_CSV, PROCESSED_CSV)
    import pandas as pd
    df = pd.read_csv(PROCESSED_CSV)
    print(f'Loaded {len(df):,} rows from cached processed data.')
    print(f'  Labels:  {dict(df["label"].value_counts())}')
    print(f'  Domains: {dict(df["domain"].value_counts())}')
    print('\nTo re-process from scratch, delete the file from Drive and re-run.')
else:
    print('Running data preprocessing pipeline...\n')
    !python data_preprocessing.py
    # Save processed output to Google Drive
    if os.path.isfile(PROCESSED_CSV):
        shutil.copy2(PROCESSED_CSV, DRIVE_PROCESSED_CSV)
        print(f'\nProcessed data saved to Google Drive: {DRIVE_PROCESSED_CSV}')
    else:
        print('ERROR: Preprocessing did not produce output. Check errors above.')

In [ ]:
import pandas as pd

df = pd.read_csv(PROCESSED_CSV)
print(f'Unified dataset: {len(df):,} rows')
print(f'Columns: {list(df.columns)}')
print(f'\nLabel distribution:')
print(df['label'].value_counts().to_string())
print(f'\nDomain distribution:')
print(df['domain'].value_counts().to_string())

# Show numeric feature columns (if any)
feature_cols = [c for c in df.columns if c not in ('text', 'label', 'domain')]
if feature_cols:
    print(f'\nNumeric feature columns ({len(feature_cols)}): {feature_cols[:10]}')
    if len(feature_cols) > 10:
        print(f'  ... and {len(feature_cols) - 10} more')
else:
    print('\nNo extra numeric feature columns detected (text-only mode).')

print('\nSample rows:')
df.head(3)

## Step 5 — Train the Model

Trains `OptimizedMultichannelCNN` on the GPU using the updated architecture:
- **Multi-channel Conv1D** (kernel sizes 2, 3, 5) with min_len trimming
- **Multi-head self-attention** for sequence pooling
- **Auxiliary numeric features** (if present in the dataset)
- **Stratified train/val split** by label + domain
- **Early stopping** + **threshold calibration** for optimal F1

The best checkpoint (highest validation F1) is saved to both the
local workspace and Google Drive.

**Key flags for reducing false positives (all on by default):**
- `--class-weighted` — inverse-frequency class weights; corrects the ~2:1 label imbalance
  (stress:no-stress) so the model doesn't learn to over-predict stress
- `--max-fpr 0.20` — threshold calibration rejects any decision threshold that causes
  FPR > 20 % on the validation set
- `--fp-penalty 0.2` — adds a soft false-positive penalty directly to the training loss
  every batch, penalising the model when it assigns high stress probability to
  non-stressed samples

**Other tunable flags:**
- `--epochs` — more epochs = potentially better accuracy but longer runtime
- `--batch-size` — reduce to 32 if you hit GPU out-of-memory errors
- `--lr` — initial learning rate (a cosine scheduler reduces it automatically)
- `--patience` — early stopping after N epochs without F1 improvement
- `--label-smoothing` — reduces overconfident predictions (0.05 is a good starting point)
- `--fp-penalty` — increase (e.g. `0.4`) if the happy/neutral FP rate is still high


In [ ]:
LOCAL_CHECKPOINT = os.path.join(REPO_DIR, 'checkpoints', 'model.pt')
DRIVE_CHECKPOINT = os.path.join(DRIVE_CHECKPOINTS, 'model.pt')

!python training/train.py \\
    --epochs 15 \\
    --batch-size 64 \\
    --lr 1e-3 \\
    --weight-decay 1e-4 \\
    --dropout 0.3 \\
    --patience 3 \\
    --class-weighted \\
    --max-fpr 0.20 \\
    --fp-penalty 0.2 \\
    --label-smoothing 0.05 \\
    --device cuda \\
    --data {PROCESSED_CSV} \\
    --output {LOCAL_CHECKPOINT}


In [ ]:
# Save the trained checkpoint to Google Drive (persistent storage)
import shutil

if os.path.isfile(LOCAL_CHECKPOINT):
    shutil.copy2(LOCAL_CHECKPOINT, DRIVE_CHECKPOINT)
    local_mb = os.path.getsize(LOCAL_CHECKPOINT) / 1e6
    print(f'Checkpoint saved to Google Drive: {DRIVE_CHECKPOINT} ({local_mb:.1f} MB)')
    print('This file persists even after the Colab session ends.')
else:
    print('ERROR: No checkpoint found. Check training output above.')

In [ ]:
import torch

ckpt_path = DRIVE_CHECKPOINT if os.path.isfile(DRIVE_CHECKPOINT) else LOCAL_CHECKPOINT

if os.path.isfile(ckpt_path):
    size_mb = os.path.getsize(ckpt_path) / 1e6
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    state = ckpt.get('model_state_dict', ckpt)
    n_params = sum(v.numel() for v in state.values())
    print(f'Checkpoint: {ckpt_path}  ({size_mb:.1f} MB)')
    print(f'Parameter count: {n_params:,}')
    print(f'Model type: {ckpt.get("model_type", "cnn")}')
    print(f'Decision threshold: {ckpt.get("decision_threshold", 0.5):.3f}')
    if 'feature_dim' in ckpt:
        print(f'Auxiliary feature dim: {ckpt["feature_dim"]}')
    if 'feature_columns' in ckpt:
        cols = ckpt['feature_columns']
        if len(cols) > 5:
            print(f'Feature columns ({len(cols)}): {cols[:5]} ...')
        else:
            print(f'Feature columns: {cols}')
    print(f'State dict keys: {list(state.keys())[:5]} ...')
else:
    print('ERROR: checkpoint not found — check training output above.')

## Step 6 — Download the Trained Checkpoint

Your checkpoint is already saved to Google Drive at:
```
MyDrive/StressDetection/checkpoints/model.pt
```

You can access it anytime from drive.google.com or any future Colab session.

**Option A — Download directly to your computer:**

In [ ]:
from google.colab import files

download_path = DRIVE_CHECKPOINT if os.path.isfile(DRIVE_CHECKPOINT) else LOCAL_CHECKPOINT

if os.path.isfile(download_path):
    files.download(download_path)
    print('Download started — check your browser downloads folder.')
else:
    print('ERROR: No checkpoint found to download.')

**Option B — Already on Google Drive!**

If you used the steps above, the checkpoint is already persisted at
`MyDrive/StressDetection/checkpoints/model.pt`.

To download it from Drive on your Windows machine:
1. Go to [drive.google.com](https://drive.google.com)
2. Navigate to `StressDetection/checkpoints/`
3. Right-click `model.pt` → Download

## Resuming a Previous Session

If your Colab session disconnected or you closed the browser:

1. **Re-run Step 1** (Mount Google Drive) — your data is still there
2. **Re-run Step 2** (Clone/pull repo + install deps)
3. **Skip Step 3** (datasets are already on Drive)
4. **Re-run Step 4** — it auto-detects the cached processed CSV on Drive
5. **Re-run Step 5** to continue training, or go straight to Step 6 to
   download your existing checkpoint

All your data is safe on Google Drive!

---
## Optional — Quick Inference Test (on Colab)

Verify the trained model works before downloading it.

In [ ]:
import hashlib
import torch
from models.architecture import OptimizedMultichannelCNN

VOCAB_SIZE = 10_000
CHUNK_SIZE = 200

def tokenize(text: str) -> list[int]:
    """Identical to _simple_tokenize in api/main.py."""
    tokens = text.lower().split()
    ids = [
        int(hashlib.md5(t.encode('utf-8'), usedforsecurity=False).hexdigest(), 16)
        % (VOCAB_SIZE - 1) + 1
        for t in tokens
    ]
    if len(ids) < CHUNK_SIZE:
        ids = ids + [0] * (CHUNK_SIZE - len(ids))
    else:
        ids = ids[:CHUNK_SIZE]
    return ids

# Load checkpoint
ckpt_path = DRIVE_CHECKPOINT if os.path.isfile(DRIVE_CHECKPOINT) else LOCAL_CHECKPOINT
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=True)
state = ckpt.get('model_state_dict', ckpt)
feature_dim = int(ckpt.get('feature_dim', 0))

model = OptimizedMultichannelCNN(
    vocab_size=VOCAB_SIZE, embed_dim=128, num_filters=64,
    kernel_sizes=(2, 3, 5), num_classes=2, dropout=0.3,
    aux_dim=feature_dim,
)
model.load_state_dict(state)
model.eval()

# Run inference on a few samples
samples = [
    'I feel completely overwhelmed and cannot stop worrying about everything.',
    'Had a great weekend hiking with friends, feeling refreshed and happy.',
    'Deadlines are piling up, I have not slept properly in days.',
    'Just got promoted at work, celebrating with family tonight!',
]

threshold = float(ckpt.get('decision_threshold', 0.5))
print(f'Decision threshold: {threshold:.3f}')
print()
print(f'{"Text":<60} {"Stress":>6}  {"Label"}')
print('-' * 80)
for text in samples:
    ids = tokenize(text)
    tensor = torch.tensor([ids], dtype=torch.long)
    with torch.no_grad():
        if feature_dim > 0:
            aux = torch.zeros((1, feature_dim), dtype=torch.float)
            out = model(tensor, aux_features=aux)
        else:
            out = model(tensor)
    prob = torch.softmax(out['logits'], dim=-1)[0, 1].item()
    label = 'stress' if prob >= threshold else 'no_stress'
    print(f'{text[:60]:<60} {prob:>6.1%}  {label}')

---
## What's Saved on Google Drive

After running this notebook, your Google Drive contains:

```
MyDrive/StressDetection/
├── data/
│   ├── raw/               <- Your original dataset files
│   └── processed/
│       └── unified_stress.csv  <- Merged & cleaned dataset
├── checkpoints/
│   └── model.pt           <- Trained model checkpoint
└── logs/                  <- (reserved for future training logs)
```

All files persist across Colab sessions. To retrain, simply re-run
the notebook — it detects existing files and skips redundant steps.

## What's Next — Running on Windows 11

You now have `model.pt` on Google Drive. Follow these steps on your Windows 11 machine:

```cmd
:: 1. Clone the repo
git clone https://github.com/anant-925/StressDetection.git
cd StressDetection

:: 2. Create and activate a virtual environment
python -m venv venv
venv\Scripts\activate
pip install --upgrade pip
pip install -r requirements.txt

:: 3. Place the downloaded checkpoint
mkdir checkpoints
copy %USERPROFILE%\Downloads\model.pt checkpoints\model.pt

:: 4. (Optional) Set security keys for production
set JWT_SECRET_KEY=your-random-secret-key-here
set FERNET_KEY=your-base64-fernet-key-here

:: 5. Start the FastAPI backend (Terminal 1)
uvicorn api.main:app --host 0.0.0.0 --port 8000

:: 6. Start the Streamlit UI (Terminal 2)
venv\Scripts\activate
streamlit run ui/app.py

:: 7. Open http://localhost:8501 in your browser
```

Or use the **one-click launcher**: double-click `run_windows.bat`

See the repository **README.md** for the complete Windows deployment guide.